# Olist data exploration

This notebook profiles every CSV in `data/raw/` without changing source data. It focuses on schema, missingness, duplicates, identifiers, timestamps, numeric distributions, and referential consistency needed for the Phase 1 PostgreSQL model.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

candidate_roots = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((root for root in candidate_roots if (root / 'data' / 'raw').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the project root or notebooks directory.')

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
csv_paths = sorted(RAW_DIR.glob('*.csv'))
if not csv_paths:
    raise FileNotFoundError(f'No CSV files found in {RAW_DIR}. Add the Olist CSVs first.')

datasets = {path.name: pd.read_csv(path, low_memory=False) for path in csv_paths}
print(f'Loaded {len(datasets)} CSV files from {RAW_DIR}')

## Structural profile

Shapes and schemas establish the expected load contract. Missing values are reported rather than filled because null timestamps and comments often carry business meaning.

In [ ]:
for name, frame in datasets.items():
    print('\n' + '=' * 90)
    print(f'{name}: shape={frame.shape}; exact_duplicate_rows={frame.duplicated().sum():,}')
    print('Columns:', frame.columns.tolist())
    display(pd.DataFrame({'dtype': frame.dtypes.astype(str), 'missing_count': frame.isna().sum(), 'missing_pct': (100 * frame.isna().mean()).round(2)}))

## Identifier checks

Primary-key candidates should be unique and non-null. Repeated foreign keys are expected in child tables. Reviews and geolocation deserve special care because their apparent identifiers are not guaranteed to be unique at row grain.

In [ ]:
id_rows = []
for name, frame in datasets.items():
    id_columns = [column for column in frame.columns if column == 'order_item_id' or column.endswith('_id')]
    for column in id_columns:
        id_rows.append({
            'dataset': name,
            'column': column,
            'rows': len(frame),
            'non_null': int(frame[column].notna().sum()),
            'distinct': int(frame[column].nunique(dropna=True)),
            'duplicate_non_null_values': int(frame[column].dropna().duplicated().sum()),
        })
display(pd.DataFrame(id_rows).sort_values(['column', 'dataset']).reset_index(drop=True))

## Timestamp discovery and ranges

Columns containing `date` or `timestamp` are parsed only for profiling. Parse failures flag malformed non-null values; source frames remain unchanged in this notebook.

In [ ]:
timestamp_rows = []
for name, frame in datasets.items():
    candidates = [column for column in frame.columns if 'date' in column.lower() or 'timestamp' in column.lower()]
    for column in candidates:
        parsed = pd.to_datetime(frame[column], errors='coerce')
        timestamp_rows.append({
            'dataset': name,
            'column': column,
            'non_null_source': int(frame[column].notna().sum()),
            'parse_failures': int((frame[column].notna() & parsed.isna()).sum()),
            'minimum': parsed.min(),
            'maximum': parsed.max(),
        })
display(pd.DataFrame(timestamp_rows))

## Numeric summaries

Distribution summaries help select PostgreSQL precision and reveal implausible extremes without automatically treating them as errors.

In [ ]:
for name, frame in datasets.items():
    numeric = frame.select_dtypes(include='number')
    if not numeric.empty:
        print(f'\n{name}')
        display(numeric.describe().T)

## Referential consistency

Each check counts distinct non-null child keys missing from its intended parent. Zero supports an enforced foreign key. Nonzero results should be investigated before adding a constraint; category translations are intentionally treated as optional coverage.

In [ ]:
relationships = [
    ('olist_orders_dataset.csv', 'customer_id', 'olist_customers_dataset.csv', 'customer_id'),
    ('olist_order_items_dataset.csv', 'order_id', 'olist_orders_dataset.csv', 'order_id'),
    ('olist_order_items_dataset.csv', 'product_id', 'olist_products_dataset.csv', 'product_id'),
    ('olist_order_items_dataset.csv', 'seller_id', 'olist_sellers_dataset.csv', 'seller_id'),
    ('olist_order_payments_dataset.csv', 'order_id', 'olist_orders_dataset.csv', 'order_id'),
    ('olist_order_reviews_dataset.csv', 'order_id', 'olist_orders_dataset.csv', 'order_id'),
    ('olist_products_dataset.csv', 'product_category_name', 'product_category_name_translation.csv', 'product_category_name'),
]

relationship_rows = []
for child_name, child_key, parent_name, parent_key in relationships:
    if child_name not in datasets or parent_name not in datasets:
        relationship_rows.append({'relationship': f'{child_name}.{child_key} -> {parent_name}.{parent_key}', 'status': 'file missing', 'orphan_distinct_keys': None})
        continue
    child_values = datasets[child_name][child_key].dropna()
    parent_values = set(datasets[parent_name][parent_key].dropna())
    orphans = child_values[~child_values.isin(parent_values)]
    relationship_rows.append({
        'relationship': f'{child_name}.{child_key} -> {parent_name}.{parent_key}',
        'status': 'checked',
        'child_non_null_rows': len(child_values),
        'orphan_rows': int(len(orphans)),
        'orphan_distinct_keys': int(orphans.nunique()),
    })
display(pd.DataFrame(relationship_rows))

## Concise observations

- Null delivery timestamps can identify orders that were canceled, unavailable, or not delivered; they should remain null.
- Review titles and messages are optional user content and should not be imputed.
- `customer_id` identifies an order-facing customer record, while `customer_unique_id` is the cross-order customer identity used for repeat-purchase analysis.
- `order_items` and `payments` use composite keys because their sequence numbers are unique only within an order.
- Review IDs and geolocation ZIP prefixes require non-unique handling to preserve legitimate source observations.
- Product category translations may not cover every Portuguese category; analytics should fall back to the source category name.
- Exact duplicate removal is safe and narrow; other anomalies should be documented and investigated rather than automatically rewritten.